In [ ]:
# run this:
%pip install -e . # or !uv pip install -e .
# to install the project in editable mode (so we can use tinyhnsw visualization tools easily)

In [ ]:
from tinyhnsw import HNSWIndex
from tinyhnsw.hnsw import DEFAULT_CONFIG

import numpy as np

np.random.default_rng(seed=42)

# Create and Visualize the HNSW

Questions: 
- Node X has some connections in layer L: are these connections preserved also in layers L-1, L-2, ...? Why?
- Do the parameters M, M_max, etc, have any influence on that?

In [ ]:
import math
from tinyhnsw.visualization import visualize_hnsw_index, generate_tiny_world

config = DEFAULT_CONFIG
config.M_max = 4
config.M_max0 = 6
config.M = 4
config.m_L = 1.0 / math.log(4)

total = 20
layout = generate_tiny_world(total, 50, 50)
vectors = np.array([layout[ix] for ix in range(total)])
index = HNSWIndex(d=2, config=config)
index.add(vectors)

visualize_hnsw_index(index, layout)

# Benchmark HNSW with different configurations

In [ ]:
import tarfile
from pathlib import Path

import hnswlib


# N = 1_000_000
N = 10_000

ef_construction = 100
dim = 128

data_path = Path.cwd().parent / "data"
indexes_folder_tarfile = data_path / "indexes.tar.gz"
# indexes_folder = data_path / "indexes" / "1M"
indexes_folder = data_path / "indexes" / "10K"

### Load the SIFT10K/SIFT1M Dataset

In [ ]:
from tinyhnsw.utils import load_sift_10K, load_sift_1M

data, queries, labels = load_sift_10K(data_dir=data_path)
# data, queries, labels = load_sift_1M(data_dir=data_path)

### Create the indexes for SIFT1M (optional)

In [ ]:
if not indexes_folder.exists():
    indexes_folder.mkdir(parents=True, exist_ok=True)

    # create the indexes
    for M in [8, 16, 32]:
        index_path = indexes_folder / f"M{M}_efConstruction{ef_construction}.index"
        print(f"Creating index: {index_path.name}")
        index = hnswlib.Index(space="l2", dim=dim)  # create the index object
        index.init_index(max_elements=N, ef_construction=ef_construction, M=M)  # initialize the index
        index.add_items(data, np.arange(data.shape[0]))  # add our vectors with identifiers
        index.save_index(str(index_path.absolute()))  # save the index for later use
        print(f"Index created: {index_path}")
elif indexes_folder.exists():
    print("Indexes folder already exists!")

In [ ]:
# if you want to compress the indexes into a new archive with parallel compression (faster!)
!tar -c --use-compress-program=pigz -f /path/to/the/indexes.tar.gz /path/to/the/indexes/folder

In [ ]:
# if you need to extract the indexes archive with parallel decompression
!tar -x --use-compress-program=pigz -f /path/to/the/indexes.tar.gz

### Load the indexes

In [ ]:
indexes = {}

for M in [8, 16, 32]:
    index = hnswlib.Index(space="l2", dim=dim)
    index.load_index(str(indexes_folder / f"M{M}_efConstruction{ef_construction}.index"), max_elements=N)
    indexes[f"M{M}"] = index

### How does the efSearch parameter impact on the Recall@1?

- We will increase the efSearch parameter from 10 to 250: what do we expect? Why?
- The M parameter used during index creation impacts too?


In [ ]:
import time

ef_search_values = [10, 50, 100, 250]

records = {}

for M in [8, 16, 32]:  # for each M value
    _records = []
    for ef_search in ef_search_values:  # for each efSearch value
        index = indexes[f"M{M}"]
        index.set_ef(ef_search)

        start_t = time.time_ns()
        predicted_labels, distances = index.knn_query(queries, k=1)  # search with the queries!
        end_t = time.time_ns()

        time_t = end_t - start_t

        recall_at_1 = sum(labels == predicted_labels.reshape(1, -1)[0]) * 100 / labels.shape[0]  # compute the recall@1

        _records.append((ef_search, time_t, recall_at_1))
    records[f"M{M}"] = _records

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(9, 4))
ax_time = ax.twinx()

M_values = [8, 16, 32]
colors = ["tab:blue", "tab:orange", "tab:green"]

for M, color in zip(M_values, colors):
    ef, times, recalls = zip(*sorted(records[f"M{M}"]))
    recalls = np.array(recalls) / 100
    ax.plot(ef, recalls, "-d", color=color, linewidth=2, markersize=5)
    ax_time.plot(
        ef,
        times,
        "--o",
        color=color,
        linewidth=1.5,
        markersize=4,
        alpha=0.65,
    )

ax.set(
    xlabel="efSearch",
    ylabel="Recall@1",
    title="Recall and search time by M",
    ylim=(0, 1.02),
)
ax_time.set_ylabel("Search time (ns)")
ax.grid(axis="both", alpha=0.15)
ax.set_axisbelow(True)

for axis in (ax, ax_time):
    axis.spines["top"].set_visible(False)

handles = [Line2D([], [], color=color, linewidth=2, label=f"M = {M}") for M, color in zip(M_values, colors)] + [
    Line2D([], [], color="0.3", linestyle="-", marker="d", label="Recall@1"),
    Line2D([], [], color="0.3", linestyle="--", marker="o", label="Search time"),
]
fig.legend(
    handles=handles,
    loc="lower center",
    ncol=5,
    frameon=False,
    bbox_to_anchor=(0.5, 0),
)

fig.tight_layout(rect=(0, 0.09, 1, 1))
plt.show()

# Pre/Post Filtering

### Load the MongoDB/embedded_movies dataseet

In [ ]:
import hnswlib
import numpy as np
from datasets import load_dataset

movies = load_dataset("MongoDB/embedded_movies", split="train").filter(lambda movie: movie["plot_embedding"])

### Take a look to our data schema

In [ ]:
# Our embeddings are here:
# Vector: movies[i]["plot_embedding"]

# While the metadata are all the remaining elements
# movies[i]["genres"], movies[i]["runtime"], etc.

movies

### Index the embeddings

In [ ]:
# get all our embeddings as a single matrix
# there are some missing embeddings in the dataset
# data = np.array([movie["plot_embedding"] if movie["plot_embedding"] else [1e9] * 1536 for movie in movies])
# ids = np.array([i for i, movie in enumerate(movies)])

data = np.array([movie["plot_embedding"] for movie in movies])
ids = np.array([i for i, movie in enumerate(movies)])

num_embeddings, dim = data.shape

print(f"Total embeddings: {num_embeddings}")
print(f"Embedding size: {dim}")

In [ ]:
index = hnswlib.Index(space="l2", dim=dim)  # instantiate the index

# define the construction parameters (change them if you want)
M, ef_construction, max_elements = 8, 50, num_embeddings 

# init the index
index.init_index(M=M, ef_construction=ef_construction, max_elements=max_elements)

# ingest the embeddings
index.add_items(data, ids)

assert len(index.get_ids_list()) == ids.shape[0]  # verify if we have correctly ingested our vectors...

## Queries

We want to obtain all the results for our queries filtered by the genre "Comedy"

In [ ]:
genre = "Comedy"

In [ ]:
# identify all the non-valid elements
valid_ids = [_id for _id, movie in enumerate(movies) if genre in movie["genres"]]

valid_ratio = len(valid_ids) / len(movies) * 100

print(f"Valid movies ratio: {valid_ratio:.2f}%")

Question: is the above ratio high or low? Will that impact the search?

Visually, how are distributed the elements in that genre?

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
data_2d = pca.fit_transform(data)

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(data_2d[:, 0], data_2d[:, 1], c=["tab:red" if genre in movie["genres"] else "tab:blue" for movie in movies])
plt.show()

In [ ]:
# X: (n_movies, embedding_dimension) aligned with movies
X = np.asarray(data, dtype=np.float32)
X_unit = X / np.linalg.norm(X, axis=1, keepdims=True)


def sample_queries(num_queries: int = 100) -> np.ndarray:
    import random

    random.seed(42)
    queries = random.choices(range(len(movies)), k=num_queries)
    return np.array(queries)


def ground_truth(
    query_ids,
    genre: str,
    k: int = 10,
) -> dict[int, np.ndarray]:
    if k < 1:
        raise ValueError("k must be positive")

    eligible_ids = np.flatnonzero([genre in (movie["genres"]) for movie in movies if movie])

    truth = {}

    for query_id in query_ids:
        query_id = int(query_id)

        # Exclude the query movie itself.
        ids = eligible_ids[eligible_ids != query_id]

        distances = 1 - X_unit[ids] @ X_unit[query_id]
        order = np.argsort(distances)[:k]
        truth[query_id] = ids[order]

    return truth


def recall(
    query_ids: np.ndarray,
    predicted_labels: np.ndarray | list[list[int]],
    labels: dict[int, np.ndarray],
    k: int = 10,
) -> float:
    if k < 1:
        raise ValueError("k must be positive")
    if len(query_ids) != len(predicted_labels):
        raise ValueError("Expected one prediction row per query")

    scores = []

    for query_id, predicted in zip(query_ids, predicted_labels):
        expected = labels[int(query_id)][:k]

        # No eligible neighbors: recall is undefined for this query.
        if len(expected) == 0:
            continue

        hits = len(set(expected) & set(predicted[:k]))
        scores.append(hits / len(expected))

    return float(np.mean(scores)) if scores else float("nan")

In [ ]:
# sample the queries...
query_ids = sample_queries()

# ... and identify their ground truth for the desired genre
labels = ground_truth(query_ids, genre=genre, k=30)

# put all the query vectors in an array ready-to-use with the index...
query_vectors = np.asarray(
    [movies[q]["plot_embedding"] for q in query_ids],
    dtype=np.float32,
)

# define how many results we want to return
k = 10

print(f"Total valid queries: {len(query_ids)}")

### Query 1: no filtering

In [ ]:
# query the index to get the predictions...
predicted_labels, distances = index.knn_query(query_vectors, k=k + 1)

# ...remove the query_id from the results...
predicted_labels = [
    [int(movie_id) for movie_id in candidates if movie_id != query_id][:k]
    for query_id, candidates in zip(query_ids, predicted_labels)
]

# ... and compute the recall
rec = recall(query_ids, predicted_labels, labels, k=k)

print(f"Recall@{k} without filtering: {rec:.3f}")

### Query 2: Pre-filtering

Pre-filter *all* the elements based on condition, then search on that restricted set of vectors.

Questions:
- How can we identify all the relevant movies before the actual search?
- Do we need to modify or re-create the index?

Hints:
- `hnswlib` allows to mark/unmark elements as deleted
- A new index with this scale is easy to re-compute

In [ ]:
# identify all the non-valid elements

In [ ]:
# query the index to get the predictions...
predicted_labels, distances = index.knn_query(query_vectors, k=k + 1)

# ...remove the query_id from the results...
predicted_labels = [
    [int(movie_id) for movie_id in candidates if movie_id != query_id][:k]
    for query_id, candidates in zip(query_ids, predicted_labels)
]

# ... and compute the recall
rec = recall(query_ids, predicted_labels, labels, k=k)

print(f"Recall@{k} with pre-filtering: {rec:.3f}")

In [ ]:
# Have we modified the original index? Do we need to update it again for later use?

### Query 3: In-search-filtering

Filter elements *during* the search itself. We need to dynamically identify the relevant elements.

Questions:
- In this case, do we need a new or modified index?

Hints:
- `hnswlib` allows for user-defined functions to filter nodes


In [ ]:
# define a support mechanism for dynamic nodes filtering

In [ ]:
# query the index to get the predictions with the filter...
predicted_labels, distances = index.knn_query(query_vectors, k=k + 1)

# ...remove from the results the movie_id==query_id...
predicted_labels = [
    [int(movie_id) for movie_id in candidates if movie_id != query_id][:k]
    for query_id, candidates in zip(query_ids, predicted_labels)
]

# ... and eventually compute the recall
rec = recall(query_ids, predicted_labels, labels, k=k)

print(f"Recall@{k} with in-search filtering: {rec:.3f}")

### Query 4: Post-filtering

Filter elements *after* the search, as a post-processing operation.
First, we identify a set of nodes, then we select only those that are actually relevant for our criteria.

Questions:
- In this case, do we need a new or modified index?
- Can we keep the *k* value used until now for the search, or do we have to manipulate it?

In [ ]:
# the k value used for the kNN query is the same here?

# query the index to get the predictions with the filter...
predicted_labels, distances = index.knn_query(query_vectors, k=k)

# what do we have to do with the retrieved elements?
predicted_labels = ... # ?

# ... and eventually compute the recall
rec = recall(query_ids, predicted_labels, labels, k=k)

print(f"Recall@{k} with post-filtering: {rec:.3f}")